## Importaciones

In [1]:
!pip install kaggle


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install load_dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json
import zipfile
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

## Utilidades

In [4]:
def detect_environment():
    try:
        ipython_env = str(get_ipython())
        if 'google.colab' in ipython_env:
            return "colab"
        elif 'zmqshell' in ipython_env:
            return "jupyter"
        else:
            return "interactive"
    except NameError:
        return "script"

In [5]:
def setup_kaggle_credentials():
    env = detect_environment()

    if env == "colab":
        from google.colab import userdata
        try:
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            print("Credenciales cargadas desde Secretos de Colab.")
        except Exception:
            print("Configura KAGGLE_USERNAME y KAGGLE_KEY en los secretos de Colab.")
    else:
        try:
            load_dotenv()
        except ImportError:
            print("'python-dotenv' no está instalado. Pasando a buscar kaggle.json...")

        if not os.environ.get('KAGGLE_USERNAME') and os.path.exists('kaggle.json'):
            with open('kaggle.json', 'r') as f:
                creds = json.load(f)
                os.environ['KAGGLE_USERNAME'] = creds['username']
                os.environ['KAGGLE_KEY'] = creds['key']
            print("Credenciales cargadas desde archivo kaggle.json local.")

        elif os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
            print("Credenciales cargadas desde variables de entorno.")

    if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
        print("No se encontraron credenciales reales. Cargando valores por defecto.")
        
        os.environ['KAGGLE_USERNAME'] = "default"
        os.environ['KAGGLE_KEY'] = "default"

In [6]:

def download_kaggle_dataset(dataset_slug, download_path="data"):
    os.makedirs(download_path, exist_ok=True)

    import kaggle

    print(f"Descargando '{dataset_slug}' en '{download_path}'...")
    kaggle.api.dataset_download_files(dataset_slug, path=download_path, unzip=False)

    dataset_name = dataset_slug.split('/')[-1]
    zip_path = os.path.join(download_path, f"{dataset_name}.zip")

    return zip_path

In [7]:
def extract_zip_file(zip_path, extract_to="data"):
    print(f"Descomprimiendo '{zip_path}'...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    return extract_to

In [8]:
def get_dataset(dataset_slug, target_file, data_dir="data"):
    os.makedirs(data_dir, exist_ok=True)
    target_path = os.path.join(data_dir, target_file)

    if os.path.exists(target_path):
        print(f"El archivo '{target_path}' ya existe. Omitiendo descarga.")
        return target_path

    zip_path = download_kaggle_dataset(dataset_slug, download_path=data_dir)
    extract_zip_file(zip_path, extract_to=data_dir)

    if os.path.exists(zip_path):
        os.remove(zip_path)

    print(f"Proceso completado. Datos listos en: {target_path}")
    return target_path

## Dataframe

In [9]:
setup_kaggle_credentials()

No se encontraron credenciales reales.


In [10]:
DATASET_SLUG = 'yasserh/housing-prices-dataset'
TARGET_FILENAME = 'Housing.csv'
DATA_DIRECTORY = 'dataset_housing'

In [12]:
csv_path = get_dataset(
    dataset_slug=DATASET_SLUG,
    target_file=TARGET_FILENAME,
    data_dir=DATA_DIRECTORY
)

Descargando 'yasserh/housing-prices-dataset' en 'dataset_housing'...
Dataset URL: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset
Descomprimiendo 'dataset_housing\housing-prices-dataset.zip'...
Proceso completado. Datos listos en: dataset_housing\Housing.csv


In [ ]:
df = pd.read_csv(csv_path)
df

In [ ]:
print(f"Dimensiones del dataset: {df.shape}")

In [ ]:
df.head(5)

In [ ]:
df.sample(5)

In [ ]:
df.describe(include='all')

In [ ]:
df.info()

In [ ]:
df_num = df.select_dtypes(include=['int64', 'float64'])
df_num

In [ ]:
df_num.head(5)

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(df_num.corr(), cmap='coolwarm', linewidths=0.5, annot=True)
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
sns.scatterplot(data=df, x='area', y='price')
plt.title('Area vs Precio')
plt.xlabel('Area')
plt.ylabel('Precio')
plt.show()

In [ ]:
columns_to_encode = [
    'mainroad', 'guestroom', 'basement', 'hotwaterheating',
    'airconditioning', 'prefarea', 'furnishingstatus', 'parking'
]

In [ ]:
df_encode = pd.get_dummies(df, columns=columns_to_encode)

In [ ]:
df_encode.dtypes

In [ ]:
X = df_encode.drop('price', axis=1)
X

In [ ]:
y = df_encode['price']
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model= LinearRegression()
model.fit(x_train_scaled, y_train)

In [ ]:
y_pred= model.predict(x_test_scaled)
print(y_pred)

In [ ]:
mse= mean_squared_error(y_test, y_pred)
rmse= np.sqrt(mse)
r2= r2_score(y_test, y_pred)

print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2: {r2}')